# 03 WeightWatcher / Spectral Analysis

Analyzes spectral CSV files, alpha trajectories, target-alpha distances, and WW-PGD projection records across discovered paired runs.

In [ ]:
from pathlib import Path
import sys, json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown


def find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / 'pyproject.toml').exists() and (p / 'src' / 'wwgpt').exists():
            return p
    raise RuntimeError('Could not locate repository root')

REPO_ROOT = find_repo_root(Path.cwd())
SRC = REPO_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from wwgpt.analysis import (
    completed_runs, discover_experiment_runs, normalize_metrics, terminal_results,
    add_generalization_measures, vocab_size_from_artifacts, summary, align_curves,
    paired_curve_differences,
)

RESULTS_ROOT = Path(globals().get('RESULTS_ROOT', REPO_ROOT / 'results'))
ANALYSIS_DIR = Path(globals().get('ANALYSIS_DIR', RESULTS_ROOT / 'notebook_analysis'))
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Repository: {REPO_ROOT}')
print(f'Results root: {RESULTS_ROOT}')
print(f'Notebook outputs: {ANALYSIS_DIR}')


## Load spectral records

In [ ]:
runs=discover_experiment_runs(RESULTS_ROOT)
spectral=[]; projections=[]
for r in runs:
    s=r['artifacts'].get('spectral.csv', pd.DataFrame())
    if not s.empty:
        s=s.copy(); s['optimizer']=r['optimizer']; s['seed']=r['seed']; s['pair_id']=r['pair_id']; s['run_dir']=str(r['run_dir']); spectral.append(s)
    p=r['artifacts'].get('wwpgd_projection.csv', pd.DataFrame())
    if not p.empty:
        p=p.copy(); p['optimizer']=r['optimizer']; p['seed']=r['seed']; p['pair_id']=r['pair_id']; projections.append(p)
spectral_df=pd.concat(spectral, ignore_index=True) if spectral else pd.DataFrame()
projection_df=pd.concat(projections, ignore_index=True) if projections else pd.DataFrame()
spectral_df.to_csv(ANALYSIS_DIR/'spectral_records.csv', index=False)
projection_df.to_csv(ANALYSIS_DIR/'projection_records.csv', index=False)
print(f'spectral rows: {len(spectral_df)}; projection rows: {len(projection_df)}')
display(spectral_df.head())


## Model-level alpha trajectories

In [ ]:
if spectral_df.empty or 'alpha' not in spectral_df:
    display(Markdown('No spectral records found yet. Run experiments with spectral logging to populate this analysis.'))
else:
    x_col='tokens_seen' if 'tokens_seen' in spectral_df else 'step'
    model_alpha=(spectral_df.assign(alpha=pd.to_numeric(spectral_df['alpha'], errors='coerce'))
                 .dropna(subset=['alpha'])
                 .groupby(['run_dir','pair_id','seed','optimizer',x_col])['alpha'].mean().reset_index())
    model_alpha['alpha_distance_to_2']=(model_alpha['alpha']-2.0).abs()
    model_alpha.to_csv(ANALYSIS_DIR/'model_alpha_by_run.csv', index=False)
    display(model_alpha.head())
    plt.figure(figsize=(9,5))
    for opt,g in model_alpha.groupby('optimizer'):
        mean=g.groupby(x_col)['alpha'].mean().reset_index()
        plt.plot(mean[x_col], mean['alpha'], marker='o', label=opt)
    plt.axhline(2.0, color='black', linestyle='--', linewidth=1, label='target alpha=2')
    plt.xlabel(x_col); plt.ylabel('mean layer alpha'); plt.legend(); plt.tight_layout()
    plt.savefig(ANALYSIS_DIR/'model_alpha_trajectory.png', dpi=160)
    plt.show()


## Layer and projection summaries

In [ ]:
if not spectral_df.empty and 'alpha' in spectral_df:
    layer_summary=(spectral_df.assign(alpha=pd.to_numeric(spectral_df['alpha'], errors='coerce'), alpha_distance_to_2=lambda d: (d['alpha']-2).abs())
                   .groupby(['optimizer','layer_name'], dropna=False)
                   .agg(records=('alpha','count'), mean_alpha=('alpha','mean'), mean_distance_to_2=('alpha_distance_to_2','mean'), sd_alpha=('alpha','std'))
                   .reset_index().sort_values(['optimizer','mean_distance_to_2']))
    layer_summary.to_csv(ANALYSIS_DIR/'layer_alpha_summary.csv', index=False)
    display(layer_summary)
if projection_df.empty:
    display(Markdown('No WW-PGD projection CSV records found.'))
else:
    numeric=projection_df.select_dtypes(include='number').columns
    projection_summary=projection_df.groupby('optimizer')[list(numeric)].agg(['count','mean','std','min','max']) if 'optimizer' in projection_df else projection_df[list(numeric)].agg(['count','mean','std','min','max'])
    projection_summary.to_csv(ANALYSIS_DIR/'projection_summary.csv')
    display(projection_summary)
